# Lab 4 – Model Endpoint Testing

Validate the real-time endpoint that Lab 4's deploy pipeline (or Lab 3A) created
from the **approved** model package in the SageMaker AI Model Registry.

This notebook is the acceptance test for the deploy stage: it proves the endpoint
is reachable, that it honours both payload formats the handler advertises, that
its latency is sane under a small load, and that the model behind it really did
come through the registry with an `Approved` status.

## Prerequisites
- An `InService` endpoint for the bank-marketing classifier
  (Lab 3A creates one named `bank-marketing-<timestamp>`; the CI/CD deploy
  pipeline creates one from the same model package).
- The prepared test split in S3 from `lab2-data-prep/lab-2b-...` (used for the
  load test). The notebook falls back to a synthetic row if it is missing.

## What you'll do
1. Discover the endpoint — no names to paste in.
2. Send a single JSON prediction and a single CSV prediction.
3. Run a small batch and report latency percentiles.
4. Check how the endpoint handles a malformed payload.
5. Trace the endpoint back to its Model Registry package and approval status.

## Step 1: Setup and endpoint discovery

In [ ]:
# Install required packages
# This may take 1-2 minutes on first run
# Ignore dependency conflicts warnings and errors.
!pip install --upgrade pip -q

# Clean uninstall to avoid cached version conflicts
%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops -q

# Reinstall with compatible versions
%pip install --no-cache-dir "sagemaker>=3.3.1,<3.5.0" "sagemaker-serve<1.5.0" \
    "mlflow==3.4.0" "sagemaker-mlflow==0.2.0" \
    "pandas" "scikit-learn" "xgboost" "awswrangler>=3.9" -q

# Restart kernel to pick up updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import boto3
import numpy as np

# The repo-root .env written by lab0-setup is the single source of truth for
# project naming; workshop_common is the single source for the feature list.
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / "workshop_env.py").exists():
        sys.path.insert(0, str(_cand))
        REPO_ROOT = _cand
        break
else:
    raise RuntimeError("Could not locate the repo root (no workshop_env.py found).")

from workshop_common.env import load_workshop_env
from workshop_common.schema import feature_names, target_column

env = load_workshop_env()
PROJECT_NAME = os.environ.get("PROJECT_NAME", "bank-marketing-prediction")
REGION = os.environ.get("AWS_DEFAULT_REGION") or boto3.Session().region_name
DATA_PREFIX = env["DATA_PREFIX"]

FEATURES = feature_names()
TARGET = target_column()

sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

print(f"Region        : {REGION}")
print(f"Project       : {PROJECT_NAME}")
print(f"Features ({len(FEATURES)}): {', '.join(FEATURES)}")
print(f"Target        : {TARGET}")

In [ ]:
# Discover the endpoint instead of hardcoding a name. Lab 3A and the deploy
# pipeline both name the endpoint with the "bank-marketing" prefix.
ENDPOINT_PREFIX = os.environ.get("ENDPOINT_NAME_PREFIX", "bank-marketing")


def discover_endpoint(prefix=ENDPOINT_PREFIX):
    """Return the newest InService endpoint whose name starts with `prefix`."""
    explicit = os.environ.get("ENDPOINT_NAME", "").strip()
    if explicit:
        return explicit
    endpoints = sm.list_endpoints(SortBy="CreationTime", SortOrder="Descending")[
        "Endpoints"
    ]
    for ep in endpoints:
        if ep["EndpointName"].startswith(prefix) and ep["EndpointStatus"] == "InService":
            return ep["EndpointName"]
    raise RuntimeError(
        f"No InService endpoint starting with '{prefix}'. Deploy one first "
        "(lab3-model-build/lab-3a_traditional_ml_experimenation.ipynb, or the "
        "Lab 4 deploy pipeline), then re-run this cell."
    )


endpoint_name = discover_endpoint()
desc = sm.describe_endpoint(EndpointName=endpoint_name)

print(f"Endpoint : {endpoint_name}")
print(f"Status   : {desc['EndpointStatus']}")
print(f"Created  : {desc['CreationTime']}")
print(f"Config   : {desc['EndpointConfigName']}")
for variant in desc.get("ProductionVariants", []):
    print(
        f"  variant {variant['VariantName']}: "
        f"{variant.get('CurrentInstanceCount')} x {variant.get('CurrentWeight')} weight"
    )

assert desc["EndpointStatus"] == "InService", (
    f"Endpoint is {desc['EndpointStatus']}, not InService — wait for it to finish "
    "deploying before testing."
)

## Step 2: Single prediction — both payload formats

The Lab 3A inference handler accepts two content types and always answers with
the same JSON envelope:

```json
{"predictions": [0], "probabilities": {"yes": [0.07], "no": [0.93]}}
```

- `application/json` — an object of **named** features (order-independent).
- `text/csv` — one headerless row of **positional** values, in schema order.

Both are exercised below with the same client, so the two answers must agree.

In [ ]:
def invoke(payload, content_type="application/json"):
    """Invoke the endpoint and return (parsed_response, latency_ms)."""
    body = json.dumps(payload) if content_type == "application/json" else payload
    started = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name, ContentType=content_type, Body=body
    )
    latency_ms = (time.perf_counter() - started) * 1000
    return json.loads(response["Body"].read().decode()), latency_ms


def positive_probability(result):
    """Pull the positive-class probability out of the handler's response."""
    probs = result.get("probabilities", {})
    for key in ("yes", "positive", "fraud"):
        if probs.get(key):
            return probs[key][0]
    raise KeyError(f"No positive-class probability in response: {sorted(probs)}")


# A representative client, on the same scale as the label-encoded training data.
sample_client = {
    "age": 41, "job": 0, "marital": 1, "education": 6, "credit_default": 0,
    "housing": 2, "loan": 0, "contact": 1, "month": 6, "day_of_week": 1,
    "duration": 180, "campaign": 2, "pdays": 999, "previous": 0,
    "poutcome": 1, "emp_var_rate": 1.1, "cons_price_idx": 93.994,
    "cons_conf_idx": -36.4, "euribor3m": 4.857, "nr_employed": 5191.0,
}
json_payload = {f: sample_client[f] for f in FEATURES}

json_result, json_latency = invoke(json_payload)

print("JSON payload")
print(f"  raw response : {json_result}")
print(f"  prediction   : {json_result['predictions'][0]}")
print(f"  P({TARGET}=1) : {positive_probability(json_result):.4f}")
print(f"  latency      : {json_latency:.0f} ms")

# The response envelope is the contract downstream labs depend on.
assert "predictions" in json_result, "Response is missing 'predictions'"
assert "probabilities" in json_result, "Response is missing 'probabilities'"
print("\n✓ Response envelope matches the documented contract")

In [ ]:
# Same client, positional CSV. Schema order is what makes this equivalent to
# the JSON call — the handler names the columns from FEATURE_NAMES itself.
csv_row = ",".join(str(json_payload[f]) for f in FEATURES)
csv_result, csv_latency = invoke(csv_row, content_type="text/csv")

print(f"CSV payload  : {csv_row}")
print(f"  prediction : {csv_result['predictions'][0]}")
print(f"  P({TARGET}=1) : {positive_probability(csv_result):.4f}")
print(f"  latency    : {csv_latency:.0f} ms")

# Identical features in either encoding must yield an identical score.
assert csv_result["predictions"][0] == json_result["predictions"][0]
assert abs(positive_probability(csv_result) - positive_probability(json_result)) < 1e-6
print("\n✓ JSON and CSV payloads agree — feature ordering is correct")

## Step 3: Small load test

Fifty real rows from the Lab 2 test split. This is a smoke test for latency and
prediction distribution, not a benchmark — the point is to catch an endpoint that
answers but answers badly (all-one-class output, multi-second latency).

Every invocation here is also captured by the handler's SQS → Lambda → Iceberg
pipeline, so these rows show up in `inference_responses` for Lab 5.

In [ ]:
import io

import pandas as pd

from sagemaker.core.helper.session_helper import Session

N_REQUESTS = 50
# Lab 2 wrote the prepared splits to the SageMaker default bucket under DATA_PREFIX.
bucket = Session().default_bucket()
test_key = f"{DATA_PREFIX}/data/test/test.csv"

try:
    obj = boto3.client("s3", region_name=REGION).get_object(Bucket=bucket, Key=test_key)
    # Lab 2 writes target-first, headerless CSVs.
    test_df = pd.read_csv(io.BytesIO(obj["Body"].read()), header=None)
    test_df.columns = ["y"] + FEATURES
    sample = test_df.sample(n=min(N_REQUESTS, len(test_df)), random_state=42)
    actuals = sample["y"].tolist()
    print(f"Loaded s3://{bucket}/{test_key} -> {test_df.shape}, sampling {len(sample)} rows")
except Exception as exc:
    print(f"⚠ Could not read the Lab 2 test split ({exc.__class__.__name__}: {exc})")
    print("  Falling back to perturbed copies of the single sample client.")
    rng = np.random.default_rng(42)
    sample = pd.DataFrame(
        [
            {**json_payload, "age": int(rng.integers(20, 70)),
             "duration": int(rng.integers(30, 900))}
            for _ in range(N_REQUESTS)
        ]
    )
    actuals = None

latencies, predictions, probabilities, errors = [], [], [], []
for _, row in sample.iterrows():
    try:
        result, latency = invoke({f: float(row[f]) for f in FEATURES})
        latencies.append(latency)
        predictions.append(result["predictions"][0])
        probabilities.append(positive_probability(result))
    except Exception as exc:
        errors.append(f"{exc.__class__.__name__}: {exc}")

lat = np.array(latencies)
print(f"\nSent            : {len(latencies)}/{len(sample)}  ({len(errors)} errors)")
print(f"Latency ms      : p50 {np.percentile(lat, 50):.0f} | "
      f"p90 {np.percentile(lat, 90):.0f} | max {lat.max():.0f}")
print(f"Predicted {TARGET}=1 : {sum(predictions)}/{len(predictions)} "
      f"({sum(predictions) / max(len(predictions), 1) * 100:.1f}%)")
print(f"Probability     : min {min(probabilities):.4f} | max {max(probabilities):.4f}")

assert not errors, f"{len(errors)} invocation(s) failed: {errors[:3]}"
# A model that answers with a single class for every row is not usable, even
# though every call returned 200.
assert 0 < len(set(predictions)) <= 2, "Unexpected prediction values"
if len(set(predictions)) == 1:
    print("\n⚠ Every row got the same class — check the model, not the endpoint.")

if actuals is not None:
    correct = sum(int(p == a) for p, a in zip(predictions, actuals))
    print(f"\nAgreement with held-out labels: {correct}/{len(actuals)} "
          f"({correct / len(actuals) * 100:.1f}%)")
    print("(A 50-row sample — indicative only. Lab 3A reports the real metrics.)")

## Step 4: Malformed payload behaviour

Worth knowing what your endpoint does with bad input, because the answer here is
not "it rejects it". The handler coerces unparseable values to `NaN` and fills
any feature the request left out with `0.0`, so a short or garbled row still
returns `200` with a prediction computed from mostly-default features.

That is a deliberate robustness choice, and also a monitoring problem: silent
defaults are invisible in endpoint error metrics. Lab 5B revisits this — the
place to catch it is input validation upstream of the endpoint, or a data-quality
check on the captured features.

In [ ]:
from botocore.exceptions import ClientError

probes = [
    ("too few columns", "56,1,1,1,0", "text/csv"),
    ("non-numeric values", "bad,data,wrong,feature,count", "text/csv"),
    ("unknown feature names", {"not_a_feature": 1.0, "age": 41}, "application/json"),
    ("empty body", "", "text/csv"),
]

for label, payload, content_type in probes:
    try:
        result, latency = invoke(payload, content_type=content_type)
        print(f"{label:24s} -> 200 OK   prediction={result['predictions'][0]} "
              f"P={positive_probability(result):.4f}  ({latency:.0f} ms)")
    except ClientError as exc:
        code_ = exc.response["Error"]["Code"]
        print(f"{label:24s} -> {code_}: {exc.response['Error']['Message'][:90]}")
    except Exception as exc:
        print(f"{label:24s} -> {exc.__class__.__name__}: {str(exc)[:90]}")

print("\nTakeaway: most malformed input is absorbed, not rejected. Validate before")
print("the endpoint if a wrong-shaped request should be a visible failure.")

## Step 5: Trace the endpoint back to the Model Registry

The deploy stage's governance claim is that *only an approved model reaches an
endpoint*. This walks the chain that proves it for the running endpoint:

`endpoint → endpoint config → model → model package → approval status`

In [ ]:
config = sm.describe_endpoint_config(EndpointConfigName=desc["EndpointConfigName"])
model_name = config["ProductionVariants"][0]["ModelName"]
model = sm.describe_model(ModelName=model_name)

container = (model.get("PrimaryContainer") or model["Containers"][0])
package_arn = container.get("ModelPackageName")

print(f"Endpoint       : {endpoint_name}")
print(f"Endpoint config: {desc['EndpointConfigName']}")
print(f"Model          : {model_name}")

if not package_arn:
    print("\n⚠ This model was built from a raw artifact URI, not a Model Registry")
    print("  package, so there is no approval status to check:")
    print(f"    ModelDataUrl = {container.get('ModelDataUrl')}")
else:
    package = sm.describe_model_package(ModelPackageName=package_arn)
    print(f"Model package  : {package_arn.split('/')[-1]}")
    print(f"Package group  : {package['ModelPackageGroupName']}")
    print(f"Approval status: {package['ModelApprovalStatus']}")

    metrics = (package.get("ModelMetrics") or {}).get("ModelQuality", {})
    if metrics.get("Statistics", {}).get("S3Uri"):
        print(f"Quality report : {metrics['Statistics']['S3Uri']}")

    for key, value in (package.get("CustomerMetadataProperties") or {}).items():
        print(f"  metadata     : {key} = {value}")

    assert package["ModelApprovalStatus"] == "Approved", (
        f"Endpoint is serving a model package in state "
        f"'{package['ModelApprovalStatus']}' — deployment gate was bypassed."
    )
    print("\n✓ Endpoint is serving an Approved registry model")

## Summary

What this run established:

- The endpoint was **discovered**, not pasted in, so the notebook works against
  whichever endpoint the deploy pipeline last produced.
- **Both payload formats** (`application/json` named features, `text/csv`
  positional row) return the documented envelope and agree with each other.
- A **50-row load test** passed with latency percentiles and a sane spread of
  predicted classes and probabilities.
- **Malformed input is absorbed, not rejected** — a finding to carry into Lab 5,
  not a bug to fix at the endpoint.
- The running endpoint traces back through its endpoint config and model to an
  **`Approved` Model Registry package**, which is the governance property the
  deploy stage exists to guarantee.

Every invocation above was also captured to `inference_responses` by the
handler, so Lab 5 (`lab5-monitoring/`) has traffic to monitor.

> Leave the endpoint running if you are going on to Lab 5 — it needs a live
> endpoint. Lab 3A's final cell deletes the endpoint, config, and model when you
> are done.